# 12 · Experiments that did not help, and diagnostics

Everything here is a **negative or neutral result**. It is kept because it explains *why* the final model looks the way it does.

**Needs:** `features3/5/6.parquet` (03), `oof_f2_l5.npy` (04). **Writes:** `v3_gam_ix`, `v3_gam_ms4`, `v3_gam_f6`, `v3_woe_*`. **Time:** about 45 minutes.

| Experiment | Result |
|---|---|
| Adversarial validation (can a model tell train from test patients?) | AUC 0.4998, so no shift; cross-validation is trustworthy |
| GAM on 762 explicit interaction terms | 0.7513, *worse* than the plain GAM (0.7524) |
| Multi-seed averaged GAM (4 seeds) | 0.7522, no gain |
| Weight-of-evidence scorecard + logistic regression | 0.7441 |
| GAM on visit-acceleration + lab-ordering features | 0.7522, no gain (LightGBM comparison never finished) |
| Residual analysis | no leftover pattern by hospital, attending, ZIP, time of day or any feature |

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

### 12.1 Adversarial validation

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def _section():
    W=WORK; D=DATA
    X=pd.read_parquet(f'{W}/features3.parquet'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
    for c in [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and c!='encounter_id']: X[c]=X[c].astype('category')
    X['is_test']=X.encounter_id.isin(te.encounter_id).astype(int)
    feats=[c for c in X.columns if c not in('encounter_id','is_test')]; y=X.is_test.values; oof=np.zeros(len(X))
    p=dict(objective='binary',learning_rate=0.05,num_leaves=31,feature_fraction=0.5,verbose=-1,n_jobs=2)
    for a,b in StratifiedKFold(5,shuffle=True,random_state=0).split(X,y):
        m=lgb.train(p,lgb.Dataset(X.iloc[a][feats],y[a]),300); oof[b]=m.predict(X.iloc[b][feats])
    print('adversarial AUC (0.5 = train and test indistinguishable): %.4f'%roc_auc_score(y,oof))
    imp=pd.Series(m.feature_importance('gain'),index=feats).sort_values(ascending=False); print('top drift features:',imp.head(5).round(0).to_dict())

_section()

### 12.2 Residual analysis: is there structure the model has not explained?

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def _section():
    X=pd.read_parquet(f'{WORK}/features2.parquet'); tr=pd.read_csv(f'{DATA}/train.csv')
    ie=pd.read_csv(f'{DATA}/index_encounters.csv')[['encounter_id','attending_id']]
    pt=pd.read_csv(f'{DATA}/patients.csv'); 
    T=X.merge(tr,on='encounter_id').merge(ie,on='encounter_id')
    T['zip3']=pd.read_csv(f'{DATA}/index_encounters.csv')[['encounter_id','patient_id']].merge(pt[['patient_id','zip3']],on='patient_id').set_index('encounter_id').zip3.reindex(T.encounter_id).values
    p=np.load(f'{WORK}/oof_f2_l5.npy'); y=T.readmit_30d.values
    T['p']=p; T['r']=y-p; T['v']=p*(1-p)
    def chi(c,minn=30):
        g=T.groupby(c,dropna=False).agg(r=('r','sum'),v=('v','sum'),n=('r','size')); g=g[g.n>=minn]
        x=(g.r**2/g.v).sum(); df=len(g); return f'{c}: chi2={x:.0f} df={df} z={(x-df)/np.sqrt(2*df):.1f}'
    for c in ['hospital_id','attending_id','zip3','attending_specialty','admit_month','admit_dow','dc_dow','dc_hour','admit_hour','admit_year','prim_code','discharge_disposition','race_ethnicity','language','marital_status','insurance_type','sex','first_unit','proc_code','n_dx','pu_total','dp_followup_scheduled_days','n_med','service_line']:
        print(chi(c))
    print('calibration by decile'); print(T.groupby(pd.qcut(T.p,10)).agg(p=('p','mean'),y=('readmit_30d','mean')))
    # hospital x service
    print(chi_h:=chi(['hospital_id','service_line'],50))
    # residual vs numeric features: correlation
    num=[c for c in T.columns if pd.api.types.is_numeric_dtype(T[c]) and c not in('r','p','v','readmit_30d')]
    cor={c:T[c].corr(T.r,method='spearman') for c in num if T[c].notna().sum()>5000}
    print(pd.Series(cor).abs().sort_values(ascending=False).head(15))

_section()

### 12.3 Why the earlier Spearman check looked alarming (it was an artifact)
The residual `y - p` takes two very different shapes for y=0 and y=1, so its Spearman correlation with any feature mostly mirrors the ranking of `p`. The Pearson correlation is about zero.

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score

def _section():
    X=pd.read_parquet(f'{WORK}/features2.parquet'); tr=pd.read_csv(f'{DATA}/train.csv')
    T=X.merge(tr,on='encounter_id'); y=T.readmit_30d.values; p=np.load(f'{WORK}/oof_f2_l5.npy')
    print('auc',roc_auc_score(y,p))
    r=y-p
    for c in ['egfr_first','lab_creatinine_first','age','lace']:
        s=T[c]; m=s.notna()
        print(c,'pearson r~x',np.corrcoef(s[m],r[m])[0,1],'spearman',pd.Series(r[m]).corr(pd.Series(s[m].values),method='spearman'),'| y~x',np.corrcoef(s[m],y[m])[0,1])

_section()

### 12.4 GAM variants on interaction / new features

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb, sys, time, json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def main(part):
    W=WORK; D=DATA
    drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
    ff='features5.parquet' if part.startswith('ix') else ('features6.parquet' if part.startswith('f6') else 'features3.parquet')
    X=pd.read_parquet(f'{W}/{ff}'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
    catc=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and c!='encounter_id']
    for c in catc: X[c]=X[c].astype('category')
    T=X.merge(tr,on='encounter_id'); S=te.merge(X,on='encounter_id',how='left'); y=T.readmit_30d.values
    assert S.encounter_id.tolist()==te.encounter_id.tolist()
    feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
    print(part,'features:',len(feats),flush=True)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)); t0=time.time()
    base=dict(objective='binary',num_leaves=2,max_depth=1,learning_rate=0.1,min_child_samples=100,
              feature_fraction=0.5,bagging_fraction=0.8,bagging_freq=1,lambda_l2=20,max_bin=63,
              verbose=-1,n_jobs=2,cat_smooth=50,max_cat_to_onehot=8)
    def run(p,nseed,tag):
        o=np.zeros(len(T)); t=np.zeros(len(S))
        for f,(a,b) in enumerate(folds):
            for s in range(nseed):
                m=lgb.train({**p,'seed':100*s+f,'bagging_seed':7*s+f,'feature_fraction_seed':13*s+f},
                            lgb.Dataset(T.loc[a,feats],y[a]),6000,valid_sets=[lgb.Dataset(T.loc[b,feats],y[b])],
                            callbacks=[lgb.early_stopping(200,verbose=False)])
                o[b]+=m.predict(T.loc[b,feats],num_iteration=m.best_iteration)/nseed
                t+=m.predict(S[feats],num_iteration=m.best_iteration)/(5*nseed)
            print(f'  {tag} fold {f} {roc_auc_score(y[b],o[b]):.5f} t={time.time()-t0:.0f}',flush=True)
        np.save(f'{W}/v3_{tag}_oof.npy',o); np.save(f'{W}/v3_{tag}_test.npy',t)
        print(f'{tag} OOF AUC {roc_auc_score(y,o):.5f} t={time.time()-t0:.0f}',flush=True)
    if part=='ix':   run(base,1,'gam_ix')
    if part=='ix_ms':run(base,3,'gam_ix_ms3')
    if part=='ix_ff':run({**base,'feature_fraction':0.15,'learning_rate':0.06},1,'gam_ix_ff15')
    if part=='ms3':  run(base,4,'gam_ms4')
    if part=='f6':   run(base,1,'gam_f6')

In [ ]:
main('ix'); main('ms3'); main('f6')   # interaction GAM, 4-seed GAM, GAM on features6

### 12.5 Weight-of-evidence scorecard

In [ ]:
import pandas as pd, numpy as np, sys, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

def _section():
    W=WORK; D=DATA
    drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
    X=pd.read_parquet(f'{W}/features3.parquet'); tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
    catc=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and c!='encounter_id']
    T=X.merge(tr,on='encounter_id'); S=te.merge(X,on='encounter_id',how='left'); y=T.readmit_30d.values
    feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
    num=[c for c in feats if c not in catc and T[c].notna().sum()>1000 and T[c].nunique()>4]
    cat=[c for c in feats if c in catc]
    print('WOE on',len(num),'numeric +',len(cat),'categorical',flush=True)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)); t0=time.time()
    NB=12
    def woe_fit(col_tr, ytr, edges=None, levels=None):
        """returns a mapping function; all statistics computed on the training part only"""
        if edges is not None:
            b=np.digitize(col_tr.values.astype(float),edges); b=np.where(pd.isna(col_tr.values),-1,b)
        else:
            b=pd.Categorical(col_tr,categories=levels).codes
        P,N=max(ytr.sum(),1),max((1-ytr).sum(),1); tab={}
        for k in np.unique(b):
            m=b==k; p=ytr[m].sum(); n=m.sum()-p
            tab[k]=np.log(((p+10)/ (P+10*len(np.unique(b)))) / ((n+10)/(N+10*len(np.unique(b)))))
        return tab
    def woe_apply(col, tab, edges=None, levels=None):
        if edges is not None:
            b=np.digitize(col.values.astype(float),edges); b=np.where(pd.isna(col.values),-1,b)
        else:
            b=pd.Categorical(col,categories=levels).codes
        return np.array([tab.get(k,0.0) for k in b],dtype=np.float32)
    oof=np.zeros(len(T)); tst=np.zeros(len(S)); oof_sum=np.zeros(len(T)); tst_sum=np.zeros(len(S))
    for f,(a,b) in enumerate(folds):
        Wa=np.zeros((len(a),len(num)+len(cat)),dtype=np.float32); Wb=np.zeros((len(b),Wa.shape[1],),dtype=np.float32); Ws=np.zeros((len(S),Wa.shape[1]),dtype=np.float32)
        ya=y[a]
        for j,c in enumerate(num):
            col=T[c]; q=np.unique(np.nanquantile(col.iloc[a].values.astype(float),np.linspace(0,1,NB+1)[1:-1]))
            tab=woe_fit(col.iloc[a],ya,edges=q)
            Wa[:,j]=woe_apply(col.iloc[a],tab,edges=q); Wb[:,j]=woe_apply(col.iloc[b],tab,edges=q); Ws[:,j]=woe_apply(S[c],tab,edges=q)
        for j2,c in enumerate(cat):
            j=len(num)+j2; lv=sorted(T[c].dropna().astype(str).unique())
            colA=T[c].iloc[a].astype(str); tab=woe_fit(colA,ya,levels=lv)
            Wa[:,j]=woe_apply(colA,tab,levels=lv); Wb[:,j]=woe_apply(T[c].iloc[b].astype(str),tab,levels=lv); Ws[:,j]=woe_apply(S[c].astype(str),tab,levels=lv)
        m=LogisticRegression(C=0.05,max_iter=400).fit(Wa,ya)
        oof[b]=m.predict_proba(Wb)[:,1]; tst+=m.predict_proba(Ws)[:,1]/5
        oof_sum[b]=Wb.sum(1); tst_sum+=Ws.sum(1)/5     # the plain unweighted scorecard
        print(f'  woe fold {f} LR {roc_auc_score(y[b],oof[b]):.5f} | raw sum {roc_auc_score(y[b],oof_sum[b]):.5f} t={time.time()-t0:.0f}',flush=True)
    print('woe_lr OOF AUC %.5f | woe_sum OOF AUC %.5f'%(roc_auc_score(y,oof),roc_auc_score(y,oof_sum)),flush=True)
    np.save(f'{W}/v3_woe_lr_oof.npy',oof); np.save(f'{W}/v3_woe_lr_test.npy',tst)
    s=lambda v:1/(1+np.exp(-(v-v.mean())/v.std()))
    np.save(f'{W}/v3_woe_sum_oof.npy',s(oof_sum)); np.save(f'{W}/v3_woe_sum_test.npy',s(tst_sum))

_section()